# Healthcare SDG — Data Generation Notebook

Runs the Synthetic Data Generator in three sequential stages:
**Smoke Test (1,000 rows) → Scale Test (50,000 rows) → Production (1,000,000 rows)**.

Each stage is a single function call. Run cells **top to bottom** in order.

---

| Stage | Rows | Expected runtime | CSV size |
|-------|------|-----------------|----------|
| Smoke test | 1,000 | < 1 second | ~0.6 MB |
| Scale test | 50,000 | ~7 seconds | ~31 MB |
| Production | 1,000,000 | ~2.4 min\* | ~618 MB |

\*On Parallels/M4: allow ~4–5 minutes due to virtualisation overhead.

## Prerequisites

**This notebook must be saved in the same directory as the SDG engine files:**



**Disk space:** ensure at least **2 GB free** before running Stage 3.

**Dependencies:** standard library only — no pip installs required.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re
import subprocess
import sys
import time
from pathlib import Path

In [ ]:
# ── Engine location ──────────────────────────────────────────────────────────
# Both SDG files must live in the same directory as this notebook.
# The assertions below catch a wrong working directory immediately,
# before any generation is attempted.
ENGINE = "sdg_engine_1m.py"

assert Path(ENGINE).exists(), (
    f"Engine not found: {Path(ENGINE).resolve()}. "
    "Move this notebook into the 1MRecords_SDG/ directory."
)
assert Path("sdg_engine.py").exists(), (
    "sdg_engine.py (reference data) not found. "
    "Both engine files must be in the same directory as this notebook."
)
print(f"✅  Engine confirmed: {Path(ENGINE).resolve()}")

In [ ]:
ENGINE = "sdg_engine_1m.py"

def run_sdg(rows, chunk_size, output_dir, fmt="csv", compress=False):
    """
    Run the SDG engine, stream its output, and display a live progress bar.

    Non-progress output (headers, validation results, file sizes) is printed
    above the bar via tqdm.write() so nothing gets overwritten.

    No --seed is passed — the engine auto-generates one from system entropy
    on every call. The seed is printed in the engine header; note it if you
    need to reproduce a specific run (pass --seed <value> on the CLI).

    Parameters
    ----------
    rows       : int   Total encounters to generate.
    chunk_size : int   Rows per in-memory chunk; controls peak RAM.
    output_dir : str   Destination directory for generated files.
    fmt        : str   Output format(s): "csv", "ndjson", or "csv,ndjson".
    compress   : bool  Gzip all output files when True.

    Returns
    -------
    bool  True if all 9 statistical validation checks passed.
    """
    from tqdm.auto import tqdm

    cmd = [
        sys.executable, ENGINE,
        "--rows",       str(rows),
        "--chunk-size", str(chunk_size),
        "--output-dir", output_dir,
        "--format",     fmt,
        "--validate",
    ]
    if compress:
        cmd.append("--compress")

    t0      = time.perf_counter()
    current = 0

    # tqdm.auto selects a Jupyter HTML widget in notebooks
    # and a plain text bar in a terminal — no environment detection needed.
    bar = tqdm(
        total=rows,
        unit="rows",
        unit_scale=True,
        unit_divisor=1_000,
        desc="Generating",
        colour="green",
    )

    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",   # ← explicit UTF-8 prevents Windows cp1252 misread
        bufsize=1,
    ) as proc:
        for raw in proc.stdout:
            line = raw.split(chr(13))[-1].strip()
            if not line:
                continue

            # Progress lines contain the pattern "| 50,000/1,000,000 |".
            # Parse the current row count and advance the bar by the delta.
            match = re.search(r'\|\s*([\d,]+)/[\d,]+\s*\|', line)
            if match:
                new_count = int(match.group(1).replace(',', ''))
                bar.update(new_count - current)
                current = new_count
            else:
                # Header lines, validation results, file sizes — print above the bar.
                tqdm.write(line)

    bar.update(rows - current)  # ensure bar reaches exactly 100 %
    bar.close()

    elapsed = time.perf_counter() - t0
    ok      = proc.returncode == 0
    icon    = "✅" if ok else "❌"
    label   = "ALL CHECKS PASSED" if ok else "ONE OR MORE CHECKS FAILED — review output above"

    print()
    print("─" * 60)
    print(f"{icon}  {label}")
    print(f"    Wall time : {elapsed:.1f}s")
    print("─" * 60)
    return ok


print("✅  run_sdg() ready — proceed to Stage 1.")

---
## Stage 1 — Smoke Test · 1,000 rows

Verifies the engine initialises correctly, all reference data loads without error,
and all 9 statistical validation checks pass at minimum scale.

All 1,000 rows fit in a single chunk () so generation is near-instant.

**Fix any FAIL here before proceeding to Stage 2.**

In [ ]:
run_sdg(rows=1_000, chunk_size=1_000, output_dir="./test_1k", fmt= 'csv', compress=False)

---
## Stage 2 — Scale Test · 50,000 rows

Confirms all cross-domain correlations hold at a statistically meaningful sample size.
These checks become reliable only beyond ~10K rows:

| Check | What it validates |
|-------|-------------------|
| Severity → LOS monotonic | Low < Moderate < High < Critical average LOS |
| Staffing → Incidents | Incident rate at ratio ≥ 6 exceeds ratio ≤ 4 |
| HEDIS HbA1c screening | 100% coverage for the diabetic cohort |
| Fraud upcoding rate | Within 2–8% of total encounters |

**Fix any FAIL here before proceeding to Stage 3.**

In [ ]:
run_sdg(rows=50_000, chunk_size=10_000, output_dir="./test_50k")

---
## Stage 3 — Production Run · 1,000,000 rows

Generates the full dataset consumed by the ETL pipeline.
Two formats are written simultaneously:

| File | Purpose | Approx. size |
|------|---------|-------------|
|  | SQL Server bulk import via the Python loader | ~618 MB |
|  | FHIR R4 interoperability (Encounter + Claim bundles) | ~865 MB |

> ⚠️  **Ensure ≥ 2 GB of free disk space before running.**
>
> ⏱️  **Expected runtime:** ~2.4 min native · ~4–5 min on Parallels/M4.

In [ ]:
run_sdg(rows=1_000_000, chunk_size=50000, output_dir="./output", fmt="csv,ndjson")

---
## Output Summary

Lists all generated files across the three stages with their sizes.

Only  is used by the ETL pipeline.
The  and  directories are for validation only
and can be deleted once Stage 3 completes successfully.

In [ ]:
DIVIDER = "  " + "-" * 66

print(DIVIDER)
print(f"  {"Stage":<22} {"File":<32} {"Size":>10}")
print(DIVIDER)

stages = [
    ("Smoke  (1K)",      "./test_1k"),
    ("Scale  (50K)",     "./test_50k"),
    ("Production (1M)",  "./output"),
]

for label, directory in stages:
    p = Path(directory)
    if p.exists():
        files = sorted(p.glob("encounters*"))
        if files:
            for f in files:
                size_mb = f.stat().st_size / 1024 / 1024
                print(f"  {label:<22} {f.name:<32} {size_mb:>8.1f} MB")
        else:
            print(f"  {label:<22} (directory exists, no output files found)")
    else:
        print(f"  {label:<22} (not yet generated)")

print(DIVIDER)
print()
print("  Next step  : run healthcareDataLoader.py (or the loader notebook)")
print("  Input file : ./output/encounters_1m.csv")